In [20]:
import pandas as pd

CORPUS_URL='https://github.com/Magallanes-at-UTDT/texts/raw/main/data/processed/CleanCorpus.csv'
corpus = pd.read_csv(CORPUS_URL)

In [21]:
corpus.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   party                 6 non-null      object 
 1   text_raw              6 non-null      object 
 2   n_characters          6 non-null      int64  
 3   n_words               6 non-null      int64  
 4   text_clean            6 non-null      object 
 5   n_characters_clean    6 non-null      int64  
 6   n_words_clean         6 non-null      int64  
 7   n_characters_removed  6 non-null      int64  
 8   pct_removed           6 non-null      float64
dtypes: float64(1), int64(5), object(3)
memory usage: 564.0+ bytes


In [22]:
TEXT_COL = "text_raw"
corpus["regex_text"] = corpus[TEXT_COL].apply(basic_cleaning)

In [23]:
corpus[[TEXT_COL, "regex_text"]].head()

,text_raw,regex_text
0,\n \n \n \n \nAHORA NACIÓN \nPLAN DE GOBIERNO...,ahora nación plan de gobierno 2026 2031 firmad...
1,\n1 \n \n \n \n \n \n \n \n \n \n \nPARTIDO D...,1 partido del buen gobierno 2026 2031 20 12 25...
2,\n \n \n \n \n \n \n \n \n \n \n \n \n \n \n ...,firmado digitalmente por levano gamarra edwin ...
3,...,plan de gobierno 2026 2031 1 plan de gobierno ...
4,\n1 \n \n \n \nPLAN DE GOBIERNO 2026–2031 \nP...,1 plan de gobierno 2026 2031 partido cívico ob...


In [24]:
# !python -m spacy download es_core_news_sm

In [25]:
import spacy
from spacy.lang.es.stop_words import STOP_WORDS

nlp = spacy.load(
    "es_core_news_sm",
    disable=["parser", "ner"]
)

In [26]:
import re

def normalize_tokens(text):
    """
    Tokenize and normalize Spanish text using spaCy.

    Processing steps:
      1. Tokenize the document.
      2. Remove spaces, punctuation, and stopwords.
      3. Remove one-character tokens.
      4. Keep only four-digit years; discard all other numbers.
      5. Lemmatize each token.
      6. Remove pronouns occasionally appended by spaCy's lemmatizer.
      7. Preserve repeated tokens for frequency-based analysis.
    """

    # Tokenize with spaCy
    doc = nlp(text)

    tokens = []

    for token in doc:

        # Skip spaces and punctuation
        if token.is_space or token.is_punct:
            continue

        # Skip stopwords
        if token.is_stop:
            continue

        # Skip one-character tokens
        if len(token.text) < 2:
            continue

        # Keep only four-digit years (e.g., 1993, 2025)
        # Discard all other numeric tokens
        if token.like_num:
            if not re.fullmatch(r"(18|19|20)\d{2}", token.text):
                continue

        # Lemmatize the token
        lemma = token.lemma_.lower().strip()

        # Fall back to the original token if no lemma is available
        if not lemma or lemma == "-pron-":
            lemma = token.text.lower()

        # spaCy occasionally returns lemmas such as
        # "adaptar él" or "sumar yo".
        # Keep only the lexical lemma.
        lemma = lemma.split()[0]

        # Remove the conjunction "u", which is not
        # flagged as a stopword by spaCy.
        if lemma == "u":
            continue

        # Preserve every occurrence of the lemma
        # (do NOT remove duplicates)
        tokens.append(lemma)

    return tokens

In [27]:
corpus["regex_tokens"] = corpus["regex_text"].apply(normalize_tokens)

In [10]:
corpus["regex_tokens"].iloc[0][:50]

['nación',
 'plan',
 'gobierno',
 '2026',
 '2031',
 'firmado',
 'digitalmente',
 'machuca',
 'castillo',
 'nelson',
 'gido',
 'fir',
 'hard',
 'fecha',
 '2025',
 'región',
 'conquistar',
 'mercado',
 'mundo',
 'índice',
 'presentación',
 'ii',
 'ideario',
 'ii',
 'principio',
 'ii',
 'objetivo',
 'ii',
 'visión',
 'partido',
 'ii',
 'valor',
 'nación',
 'iii',
 'visión',
 'plan',
 'gobierno',
 'iv',
 'estrategia',
 'iv',
 'dimensión',
 'social',
 'iv',
 'propuesta',
 'salud',
 'iv',
 'propuesta',
 'educación',
 'iv',
 'propuesta']

In [11]:
from sklearn.feature_extraction.text import CountVectorizer

corpus["regex_normalized_text"] = corpus["regex_tokens"].str.join(" ")

In [12]:
corpus["regex_normalized_text"].iloc[0][:500]

'nación plan gobierno 2026 2031 firmado digitalmente machuca castillo nelson gido fir hard fecha 2025 región conquistar mercado mundo índice presentación ii ideario ii principio ii objetivo ii visión partido ii valor nación iii visión plan gobierno iv estrategia iv dimensión social iv propuesta salud iv propuesta educación iv propuesta vivienda iv propuesta agua saneamiento iv propuesta fomento empleo iv propuesta inclusión social iv propuesta cultura iv dimensión institucional iv propuesta segur'

In [13]:
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

vectorizer = CountVectorizer()

dtm_sparse = vectorizer.fit_transform(
    corpus["regex_normalized_text"]
)

dtm = pd.DataFrame(
    dtm_sparse.toarray(),
    index=corpus["party"],
    columns=vectorizer.get_feature_names_out()
)

dtm.index.name = None

In [15]:
dtm

,0e5,10mil,120k,12h,1818,1824,1907,1950,1960,1970,...,óptico,óptimo,óptimos,órbita,órdén,órgano,últimamente,únicamente,único,útil
AhoraNacion,0,0,0,0,0,0,0,0,0,0,...,0,3,1,0,0,3,0,0,10,0
BuenGobierno,1,0,1,2,1,0,0,0,0,0,...,5,0,0,1,0,6,0,0,18,1
FuerzaPopular,0,0,0,0,0,0,1,0,0,0,...,3,4,0,0,1,5,0,1,24,0
JuntosPorElPeru,0,12,0,0,0,1,0,3,1,1,...,0,1,0,0,0,2,1,6,9,0
Obras,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,6,1
RenovacionPopular,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0


In [16]:
corpus.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   party                  6 non-null      object 
 1   text_raw               6 non-null      object 
 2   n_characters           6 non-null      int64  
 3   n_words                6 non-null      int64  
 4   text_clean             6 non-null      object 
 5   n_characters_clean     6 non-null      int64  
 6   n_words_clean          6 non-null      int64  
 7   n_characters_removed   6 non-null      int64  
 8   pct_removed            6 non-null      float64
 9   regex_text             6 non-null      object 
 10  regex_tokens           6 non-null      object 
 11  regex_normalized_text  6 non-null      object 
dtypes: float64(1), int64(5), object(6)
memory usage: 708.0+ bytes


In [17]:
validation = pd.DataFrame({
    "n_regex_tokens": corpus.set_index("party")["regex_tokens"].str.len(),
    "dtm_counts": dtm.sum(axis=1)
})

validation["difference"] = (
    validation["n_regex_tokens"] - validation["dtm_counts"]
)

validation

,n_regex_tokens,dtm_counts,difference
AhoraNacion,27678,27678,0
BuenGobierno,16421,16421,0
FuerzaPopular,25745,25745,0
JuntosPorElPeru,15299,15299,0
Obras,2980,2980,0
RenovacionPopular,3647,3647,0


In [18]:
term_frequency = (
    dtm.sum(axis=0)
       .sort_values(ascending=False)
)

term_frequency.head(30)

nacional           1165
público             679
desarrollo          648
sistema             641
meta                514
estratégico         477
objetivo            469
perú                468
servicio            460
salud               447
país                446
gobierno            408
acceso              402
plan                394
social              392
programa            374
nivel               354
indicador           347
gestión             343
2031                342
regional            331
infraestructura     329
agua                326
económico           321
sector              320
sostenible          318
fortalecer          316
educación           315
productivo          303
problema            295
dtype: int64

In [36]:
from pathlib import Path

PROCESSED_DIR = Path("data/processed")

dtm.to_parquet(
    PROCESSED_DIR / "DocumentTermMatrix_Regex_Counts.parquet"
)